In [ ]:
import modal

app = modal.App("rag-orchestrator")

# Definition for a lightweight CPU environment for query embedding
cpu_image = modal.Image.debian_slim().pip_install("sentence-transformers")
# Definition for a high-performance GPU environment for heavy cross-encoding
gpu_image = modal.Image.debian_slim().pip_install("torch", "transformers", "sentence-transformers")


@app.function(image=cpu_image)
def retrieve_candidates(query: str):
    # 1. Load a lightweight embedding model on CPU
    # 2. Convert query to vector
    # 3. Query your external vector database (e.g., Qdrant/Elastic)
    # 4. Return top 100 candidates
    pass


@app.function(image=gpu_image, gpu="L4")
def rerank_documents(query: str, candidates: list):
    # 1. Boot up a fast L4 GPU container
    # 2. Feed query + 100 documents into BGA-Reranker-Large
    # 3. Sort and slice down to the top 30-40 hyper-relevant documents
    pass


@app.function(image=gpu_image, gpu="A100")
def generate_response(query: str, ranked_docs: list):
    # 1. Spin up an A100 to handle the huge 20k token KV cache
    # 2. Run your generator (e.g., Llama-3-8B via vLLM)
    # 3. Stream the final answer back to your frontend
    pass


@app.function(image=cpu_image)
@modal.web_endpoint()
def orchestrate_rag_pipeline(user_query: str):
    # The clean workflow master controller
    candidates = retrieve_candidates.local(user_query)
    best_docs = rerank_documents.local(user_query, candidates)
    return generate_response.remote_gen(user_query, best_docs)

In [ ]:
# import os
# import subprocess
# import modal

# # Define the name of the model you want to deploy
# MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
# VLLM_PORT = 8000

# # 1. Define the container image with CUDA and vLLM
# vllm_image = (
#     modal.Image.from_registry("nvidia/cuda:12.9.0-devel-ubuntu22.04", add_python="3.12")
#     .uv_pip_install("vllm==0.21.0")
#     # Enable high performance downloading from Hugging Face Hub
#     .env({"HF_XET_HIGH_PERFORMANCE": "1"})
# )

# # 2. Create persistent volumes so you only download the weights ONCE
# hf_cache_vol = modal.Volume.from_name("huggingface-cache", create_if_missing=True)
# vllm_cache_vol = modal.Volume.from_name("vllm-cache", create_if_missing=True)

# # Create the Modal App
# app = modal.App("rag-vllm-backend")

# # 3. Decorate the function to act as a web server on a GPU
# @app.function(
#     image=vllm_image,
#     gpu="A10G:1", # Great cost-efficient GPU for an 8B quantized model
#     scaledown_window=5 * 60, # Scale to zero after 5 minutes of inactivity
#     timeout=15 * 60,
#     volumes={
#         "/root/.cache/huggingface": hf_cache_vol,
#         "/root/.cache/vllm": vllm_cache_vol,
#     },
#     # Pass your Hugging Face Secret if using gated models like Llama 3
#     secrets=[modal.Secret.from_name("huggingface-secret")],
# )
# @modal.web_server(port=VLLM_PORT, requires_proxy_auth=False)
# def serve():
#     cmd = [
#         "vllm", "serve", MODEL_NAME,
#         "--served-model-name", MODEL_NAME,
#         "--host", "0.0.0.0",
#         "--port", str(VLLM_PORT),
#         "--quantization", "bitsandbytes",
#         "--load-format", "bitsandbytes",
#         "--enforce-eager" # Significantly speeds up cold starts on serverless
#     ]
    
#     # Run the background server process
#     subprocess.Popen(" ".join(cmd), shell=True)

In [ ]:
# import modal
# from fastapi.responses import StreamingResponse

# app = modal.App("async-rag-pipeline")


# @app.function()
# def retrieve_docs(query: str) -> list:
#     # Standard, clean, blocking vector search logic
#     return ["doc1_chunk...", "doc2_chunk...", "etc..."]


# @app.function(gpu="L4")
# def rerank_docs(query: str, docs: list) -> list:
#     # Heavy tensor matrix calculations on GPU
#     return docs[:40]  # Returns top 40 sorted chunks


# @app.function(gpu="A100")
# async def generate_stream(query: str, context_docs: list):
#     # This must be async to yield tokens over the wire as they appear
#     # Simulated token emission from an LLM engine:
#     for token in ["This ", "is ", "the ", "grounded ", "answer..."]:
#         yield token


# @app.function()
# @modal.concurrent(max_inputs=64)  # Allows 64 users to hit this container concurrently
# @modal.web_endpoint()
# async def endpoint(user_query: str):
#     # 1. Fetch chunks (runs synchronously)
#     candidates = retrieve_docs.local(user_query)

#     # 2. Trim down to 40 (runs synchronously on GPU)
#     best_40 = rerank_docs.local(user_query, candidates)

#     # 3. Stream back to the frontend asynchronously
#     # Notice we use .remote_gen.aio() to handle the async generator stream
#     async def response_generator():
#         async for token in generate_stream.remote_gen.aio(user_query, best_40):
#             yield token

#     return StreamingResponse(response_generator(), media_type="text/plain")

In [ ]:
# import os

# import modal

# search_core_mount = modal.Mount.from_local_dir(
#     local_path="../../packages/search_core/search_core",
#     remote_path="/root/search_core"
# )

# image = (
#     modal.Image.debian_slim(python_version="3.12")
#     .pip_install("qdrant-client", "pydantic>=2.0.0", "torch", "transformers")
# )

# app = modal.App("async-rag-pipeline", image=image, mounts=[search_core_mount])

# # Core library models
# from search_core.models import SearchConfig, SearchMode, SearchQuery
# from search_core.stores import QdrantEmbeddingStore, QdrantStoreConfig


# @app.cls()  # 1. Convert to a class for container persistent state
# class DocumentRetriever:
#     @modal.enter()  # Runs ONCE when container starts up
#     def setup(self):
#         qdrant_url = os.environ.get("VECTOR_DB_URL", "http://localhost:6333")
#         config = QdrantStoreConfig(url=qdrant_url, collection_name="documents")

#         # Connection stays open and warm in memory
#         self.store = QdrantEmbeddingStore(config=config)

#     @modal.method()  # Exposes this as an invokable remote method
#     def retrieve(self, user_query: str) -> list:
#         query_obj = SearchQuery(
#             text=user_query,
#             config=SearchConfig(limit=50, mode=SearchMode.HYBRID)
#         )
#         search_response = self.store.search(query_obj)
#         return [hit.document.content for hit in search_response.results]


# @app.cls(gpu="L4")  # 2. Keep the heavy GPU model loaded in VRAM
# class DocumentReranker:
#     @modal.enter()
#     def load_model(self):
#         from search_core.rerankers import Reranker
#         # Model weights load into GPU memory ONCE
#         self.reranker = Reranker(model_name="BAAI/bge-reranker-large")

#     @modal.method()
#     def rerank(self, query: str, docs: list) -> list:
#         ranked_results = self.reranker.rerank(query=query, documents=docs)
#         return [item.document for item in ranked_results[:40]]


# @app.cls(gpu="A100")
# class LLMEngine:
#     @modal.enter()
#     def load_llm(self):
#         # TODO: Initialize your vLLM engine or weights here
#         pass

#     @modal.method()
#     async def generate_stream(self, query: str, context_docs: list):
#         simulated_tokens = ["Based ", "on ", "the ", "retrieved ", "context... "]
#         for token in simulated_tokens:
#             yield token


# # --- Orchestration Endpoint ---

# @app.function()
# @modal.concurrent(max_inputs=64)
# @modal.web_endpoint()
# async def endpoint(user_query: str):
#     # Instantiate the classes (Modal maps these to warm container instances)
#     retriever = DocumentRetriever()
#     reranker = DocumentReranker()
#     llm = LLMEngine()

#     # .remote() execution reuses the warmed up class instances
#     candidates = retriever.retrieve.remote(user_query)
#     best_40 = reranker.rerank.remote(user_query, candidates)

#     async def response_generator():
#         # Notice we use .remote_gen.aio() for async generator streams
#         async for token in llm.generate_stream.remote_gen.aio(user_query, best_40):
#             yield token

#     return StreamingResponse(response_generator(), media_type="text/plain")

In [ ]:
import subprocess
import modal

MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
VLLM_PORT = 8000

vllm_image = (
    modal.Image.from_registry("nvidia/cuda:12.4.1-devel-ubuntu22.04", add_python="3.11")
    .pip_install("vllm==0.5.4", "bitsandbytes")
)

hf_cache_vol = modal.Volume.from_name("huggingface-cache", create_if_missing=True)
app = modal.App("rag-vllm-backend")

@app.function(
    image=vllm_image,
    gpu="A10G", # Plenty for a 4-bit 8B model!
    scaledown_window=5 * 60,
    volumes={"/root/.cache/huggingface": hf_cache_vol},
    secrets=[modal.Secret.from_name("huggingface-secret")],
)
@modal.web_server(port=VLLM_PORT, requires_proxy_auth=False)
def serve():
    cmd = [
        "vllm", "serve", MODEL_NAME,
        "--served-model-name", MODEL_NAME,
        "--host", "0.0.0.0",
        "--port", str(VLLM_PORT),
        "--quantization", "bitsandbytes",
        "--load-format", "bitsandbytes",
        "--enforce-eager"
    ]
    subprocess.Popen(" ".join(cmd), shell=True)

Deploy this app by running: modal deploy vllm_app.py

In [ ]:
from fastapi.responses import StreamingResponse
import modal

app = modal.App("rag-orchestrator")

# Definition for a lightweight CPU environment
cpu_image = modal.Image.debian_slim().pip_install("sentence-transformers", "openai", "fastapi")
# Definition for a high-performance GPU environment for heavy cross-encoding
gpu_image = modal.Image.debian_slim().pip_install("torch", "transformers", "sentence-transformers")

@app.function(image=cpu_image)
def retrieve_candidates(query: str):
    # (Your existing semantic search fetching 100 documents)
    return ["Doc content A", "Doc content B"]

@app.function(image=gpu_image, gpu="L4")
def rerank_documents(query: str, candidates: list):
    # (Your existing BGA-Reranker-Large cross-encoder logic)
    return candidates[:5] # Sliced down to context winners

@app.function(image=cpu_image)
def generate_response(query: str, ranked_docs: list):
    from openai import OpenAI
    
    # 1. Grab the live URL of your deployed vLLM service
    try:
        vllm_url = modal.Server.get_url("rag-vllm-backend", "serve")
    except Exception:
        raise RuntimeError("vLLM backend is not deployed or active.")

    client = OpenAI(base_url=f"{vllm_url}/v1", api_key="not-needed")
    
    # 2. Construct the context string
    context = "\n".join([f"--- Source {i+1} ---\n{doc}" for i, doc in enumerate(ranked_docs)])
    
    messages = [
        {"role": "system", "content": "You are a concise assistant. Use the context to answer the user."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
    ]

    # 3. Stream chunks directly from the vLLM server to the generator
    response = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        messages=messages,
        max_tokens=512,
        temperature=0.6,
        stream=True # Activates token streaming
    )
    
    for chunk in response:
        if chunk.choices[0].delta.content:
            yield chunk.choices[0].delta.content

@app.function(image=cpu_image)
@modal.fastapi_endpoint() # Updated to the latest stable FastAPI decorator
def orchestrate_rag_pipeline(user_query: str):
    # Use .remote() instead of .local() so they execute in their respective cloud containers
    candidates = retrieve_candidates.remote(user_query)
    best_docs = rerank_documents.remote(user_query, candidates)
    
    # Wrap the python generator token stream into an HTTP StreamingResponse
    return StreamingResponse(
        generate_response.remote_gen(user_query, best_docs), 
        media_type="text/event-stream"
    )

Deploying `orchestrator.py` on Modal is incredibly straightforward because Modal handles the container compilation, routing, and deployment directly via the terminal.

Ensure you have deployed your `vllm_app.py` first, as the orchestrator relies on it being active in your workspace to pull its internal URL.

Here is the exact step-by-step process:

### Step 1: Install Modal Locally & Authenticate

If you haven't already, install the Modal client on your local computer and link it to your account:

```bash
pip install modal
modal setup

```

*(This will open a browser window to securely log you in and save your token config).*

### Step 2: The Two Deployment Modes

You have two distinct ways to run your `orchestrator.py` script from the terminal depending on whether you are testing it or pushing it to production.

#### Mode A: Interactive Testing (`modal serve`)

While you are developing, writing code, or tweaking your prompt formatting, do **not** do a full deployment. Instead, run:

```bash
modal serve orchestrator.py

```

* **What happens:** Modal immediately spins up your pipeline in the cloud and gives you a **temporary development URL** (e.g., `[https://yourusername--rag-orchestrator-orchestrate-rag-pipeline-dev.modal.run](https://yourusername--rag-orchestrator-orchestrate-rag-pipeline-dev.modal.run)`).
* **Live Reloading:** It keeps your terminal open. The moment you press `Ctrl+S` on your local IDE to change a line of code, Modal instantly streams the changes to the cloud container within a second.
* Hit `Ctrl+C` in your terminal to shut it down when done testing.

#### Mode B: Production Deployment (`modal deploy`)

Once your testing passes and you are ready to connect this RAG endpoint to your real frontend application or website, run:

```bash
modal deploy orchestrator.py

```

* **What happens:** Modal builds a permanent production-ready release. It will close your terminal loop and output a **permanent production URL**.
* This endpoint is now live 24/7 on the public internet. If no one uses it, the containers scale down to zero automatically so you aren't billed, but the web address stays securely active and waiting for traffic.

---

### Step 3: Testing the Endpoint Via Curl

Because we decorated the main function with `@modal.fastapi_endpoint()`, it automatically accepts basic HTTP query strings.

You can copy the URL Modal spits out into your terminal and test it right away to watch your RAG pipeline stream tokens back live:

```bash
curl -N "https://yourusername--rag-orchestrator-orchestrate-rag-pipeline.modal.run?user_query=What+is+the+capital+of+France"

```

*(The `-N` flag tells curl to disable buffering so you can see the tokens stream in one by one instantly).*